<center>

## ARHITEKTURE NEURONSKIH MREŽA
## Vježba 4 <br>**Implementacija unaprijedne neuronske mreže** "***from scratch***"

<br>

Originalni kod:
[Michael Nielsen - Neural Networks and Deep Learning](http://neuralnetworksanddeeplearning.com/)



</center>


In [ ]:
import numpy as np
import random
from time import time

`jdc` - Jupyter Dynamic Classes

In [ ]:
!pip install jdc            # instaliramo paket u colabu

In [ ]:
import jdc

# Neuronska mreža

### Aktivacijska funkcija: **sigmoidna** funkcija

Sigmoidna funkcija $\sigma: \mathbb{R} \to \mathbb{R}$ definirana  je s

$$\sigma(z) = \frac{1}{1+e^{-z}}, \quad \forall z\in\mathbb{R}.$$








<img alt = 'sigmoidna_funkcija' src = 'https://miro.medium.com/max/4384/1*6A3A_rt4YmumHusvTvVTxw.png'  width = '70%' />

In [ ]:
def sigmoid(z):
  """
  ulaz: array z
  izlaz: array istog oblika kao i z sa vrijednostima dobivenim primjenom sigmoidne funkcije po elementima
  """
  return 1.0 / (1.0 + np.exp(-z))

In [ ]:
sigmoid(np.array([1.25, 10, -5]))

array([0.77729986, 0.9999546 , 0.00669285])

##### Derivacija sigmoidne funkcije:


In [ ]:
def sigmoid_derivacija(z):
  return sigmoid(z) * (1 - sigmoid(z))

Izvod:


> $ \begin{align}
\sigma'(z) &= \left( \frac{1}{1+e^{-z}} \right)' \\
           &= -\frac{1}{(1+e^{-z})^2}\cdot (1+e^{-z})' \\
           &= -\frac{1}{(1+e^{-z})^2} \cdot e^{-z} \cdot (-1) \\
           &= \frac{e^{-z}}{(1+e^{-z})^2} \\
           &= \frac{1}{1+e^{-z}} \cdot \frac{e^{-z}}{1+e^{-z}} \\
           &= \sigma(z) \cdot \frac{e^{-z} + 1 - 1}{1+e^{-z}} \\
           &= \sigma(z) \left(1 - \sigma(z)\right)
\end{align}$




### Funkcija gubitka:  

> $L(\boldsymbol{y},\boldsymbol{\hat{y}})= - \sum\limits_{i}\big(y_i\log\hat{y_i} + (1-y_i)\log(1-\hat{y_i})\big)$


In [ ]:
class CrossEntropyCost:
    @staticmethod
    def fn(a, y):
        """Return the cost associated with an output ``a`` and desired output
        ``y``.  Note that np.nan_to_num is used to ensure numerical
        stability.  In particular, if both ``a`` and ``y`` have a 1.0
        in the same slot, then the expression (1-y)*np.log(1-a)
        returns nan.  The np.nan_to_num ensures that that is converted
        to the correct value (0.0).
        """
        return np.sum(np.nan_to_num(-y*np.log(a)-(1-y)*np.log(1-a)))

    @staticmethod
    def delta(z, a, y):
        """Return the error delta from the output layer.  Note that the
        parameter ``z`` is not used by the method.  It is included in
        the method's parameters in order to make the interface
        consistent with the delta method for other cost classes.
        """
        return (a-y)

`np.nan_to_num`: za numeričku stabilnost
  - zamijeni `nan` vrijednosti s 0, `infinity` s najvećim floating point brojem

##### Primjer

---

In [ ]:
CrossEntropyCost.fn(a = np.array([0.25, 0.25, 0.9]),
                    y = np.array([0, 0, 1]))

np.float64(0.6807246605613881)

In [ ]:
CrossEntropyCost.fn(a = np.array([0.25, 0.25, 0.5]),
                    y = np.array([0, 0, 1]))

np.float64(1.268511325463507)

## Implementacija neuronske mreže

Konstruktor klase `Network`

In [ ]:
class Network:

  def __init__(self, sizes, cost=CrossEntropyCost):
    """
    sizes - lista koja sadrži broj neurona po slojevima
            npr. [4,3,2] predstavlja mrezu koja ima tri sloja koji redom imaju 4 (ulazi), 3 i 2 neurona.
    """
    self.num_layers = len(sizes)                              # broj slojeva
    self.sizes = sizes                                        # broj neurona po sloju
    self.default_weight_initializer()                         # inicijalizacija težina
    self.cost = cost                                          # funkcija cijene koja se koristi

#### Zadana inicijalizacija težina mreže
  - za **inicijalizaciju pomaka** (***biasa***) se koristi jedinična normalna distribucija $X\sim \mathcal{N}(0,1)$
      - *biasi* (težine bridova koje povezuju odgovarajući neuron s neuronom konstantne vrijednosti 1)
  - za **inicijalizaciju težina** se koristi normalna distribucija $X\sim \mathcal{N}(0,\frac{2}{n_{in}})$ gdje je $n_{in}$ broj bridova koji povezuje dani neuron s neuronima prethodnog sloja (***He-Normalna*** inicijalizacija).


In [ ]:
%%add_to Network
def default_weight_initializer(self):
    self.biases = [np.random.randn(y, 1) for y in self.sizes[1:]]
    self.weights = [np.random.randn(y, x)*(np.sqrt(2/x))
                   for x, y in zip(self.sizes[:-1], self.sizes[1:])]

##### Primjer

---

In [ ]:
nn = Network([4,3,2])

In [ ]:
biasi = nn.biases
print('b1')
print(biasi[0])
print('\nb2')
print(biasi[1])

b1
[[0.40094103]
 [1.21728187]
 [0.73670145]]

b2
[[-0.2131987 ]
 [-0.49212969]]


In [ ]:
tezine = nn.weights
for i in range(len(tezine)):
  print(f'Tezine {i+1}:\n{tezine[i]}\n')

Tezine 1:
[[-0.11989016  0.43122906 -2.04953196  0.22985113]
 [ 0.92787304 -0.54892234 -0.39865359  0.64478691]
 [-0.06307782 -0.59573142 -0.44360679  0.5769387 ]]

Tezine 2:
[[ 0.6793587  -0.34533213 -0.23698532]
 [ 0.59931879  0.14213476  0.99715719]]



### Feedforward

> **Uočimo**: Zbog jednostavnosti u svim slojevima koristimo *sigmoidnu* aktivacijsku funkciju.

In [ ]:
%%add_to Network
def feedforward(self, a):
  """ Izracunaj predvidjanje mreze za dani ulaz a. """
  for b, w in zip(self.biases, self.weights):
    a = sigmoid(np.dot(w, a) + b)
  return a

### Evaluacija

U konkretnom primjeru bavit ćemo se problemom s prošlih vježbi:
***klasifikacijom slika znamenki iz MNIST skupa podataka***

- *Izlaz* mreže za dani ulaz $x$: `self.feedforward(x)` $\in \langle0,1\rangle^{10}$ $\implies$ predviđena znamenka  `np.argmax(self.feedforward(x))`

- Točna labela $y$ one-hot kodirana $\implies$ točna znamenka je `np.argmax(y)`.

In [ ]:
%%add_to Network
def evaluate(self, data):
  """ Metoda vraca broj primjera koji su dobro klasificirani. """
  results = [(np.argmax(self.feedforward(x)), np.argmax(y)) for (x, y) in data]
  return sum(int(x == y) for (x, y) in results)

### SGD

Treniramo neuronsku mrežu koristeći *mini-batch* verziju *stohastičkog gradijentnog spusta*.


Parametri metode `fit`:
- `training_data`: lista uređenih parova $(\boldsymbol{x}, y)$ gdje je $\boldsymbol{x}$ ulazni podatak iz skupa za treniranje, a $y$ labela tj. točna vrijednost izlaza
- `epochs`: broj epoha koji ćemo trenirati mrežu.
- `mini_batch_size`: broj podataka u mini-grupi (nakon toliko podataka ćemo ažurirati težine)
- `eta` ($\eta$): stopa učenja
- `test_data`: lista uređenih parova $(\boldsymbol{x}, y)$ gdje je $\boldsymbol{x}$ ulazni podatak iz skupa za *testiranje*, a $y$ točna vrijednost izlaza.

In [ ]:
%%add_to Network
def fit(self, training_data, epochs=30, mini_batch_size=32, eta=0.1, test_data=None):
  n = len(training_data)
  print('Treniranje mreže započelo...')
  for epoch in range(epochs):
    print('\n' + '#'*50)
    print('U tijeku EPOHA {}'.format(epoch + 1))
    random.shuffle(training_data)
    mini_batches = [training_data[k:k+mini_batch_size] for k in range(0, n, mini_batch_size)]

    for mini_batch in mini_batches:
      self.SGD_update(mini_batch, eta)

    print('accuracy treniranje {1:.2f}%'.format(epoch + 1, 100.0 * self.evaluate(train_data) / n))
    if test_data:
      print('accuracy testiranje {1:.2f}%'.format(epoch + 1, 100.0 * self.evaluate(test_data) / len(test_data)))
    else:
      print('Epoha {0} gotova.'.format(epoch + 1))

#### Ažuriranje težina



 - **Loss** (gubitak) $L$: kvantificira pogrešku na na jednom podatku
 - **Funkcija cijene** $J$: kvantificira ukupan gubitak mreže na svim podacima
 > *prosječan gubitak na danim podacima*

Označimo s $\boldsymbol{\theta}$ sve parametre mreža (težine i pomaci).

Način na koji se parametri ažuriraju koristeći metodu **gradijentnog spusta**:

$$ \boldsymbol{\theta}^{(k+1)}= \boldsymbol{\theta}^{(k)} - \eta \nabla_\theta J(\boldsymbol{\theta}^{(k)}; \mathcal{D}_{train})$$

npr. za  težinu $w_j$
$$w_j^{(k+1)} = w_j^{(k)}-\eta\frac{\partial J}{\partial w_j}\big(\boldsymbol{\theta}^{(k)}; \mathcal{D}_{train}\big).$$

- $\eta$ - stopa učenja

Kod **stohastičkog gradijentnog spusta** koristi se **procjena** gradijenta $\nabla_\theta J$ izračunata na mini-grupi $\{(\boldsymbol{x}_1, y_1), \dots, (\boldsymbol{x}_m, y_m), \} \subset \mathcal{D}_{train}$

$$\hat{\boldsymbol{g}} = \frac{1}{m} \nabla_\theta \sum\limits_{i=1}^m L(\hat{y}_i, y_i).$$

In [ ]:
%%add_to Network
def SGD_update(self, mini_batch, eta):
    """
    mini_batch: lista uredjenih parova (x,y) iz skupa za treniranje; x ulaz, y labela.
    eta: stopa ucenja
    """
    nabla_b = [np.zeros(b.shape) for b in self.biases]
    nabla_w = [np.zeros(w.shape) for w in self.weights]

    for x, y in mini_batch:
        delta_nabla_b, delta_nabla_w = self.backprop(x, y)                      # izracunaj gradijent

        nabla_b = [nb + dnb for nb, dnb in zip(nabla_b, delta_nabla_b)]
        nabla_w = [nw + dnw for nw, dnw in zip(nabla_w, delta_nabla_w)]

    self.weights = [w - (eta / len(mini_batch)) * nw
                    for w, nw in zip(self.weights, nabla_w)]
    self.biases = [b - (eta / len(mini_batch)) * nb
                   for b, nb in zip(self.biases, nabla_b)]

### Propagacija pogreške unatrag (engl. *Backpropagation*)

> metoda za računanje gradijenta za svaki podatak
> - želimo dobiti smjer za taj ulazni podatak u kojem želimo "pomaknuti" parametre mreže

- postupak se zasniva na lančanom pravilu deriviranja.

<br>
<img src ='https://drive.google.com/uc?id=1ZEj86zbqPG7P1jSOEZrYoMpI2aVcT7-s' alt='graf' width='50%'>


<br><br>
<img src ='https://drive.google.com/uc?id=1w6qjhjhFnibks8-0cj801n0qIu9bDFmp' alt='gralancano pravilo' width='50%'>


Za one koje zanimaju detalji:
- [video1](https://www.youtube.com/watch?v=Ilg3gGewQ5U&ab_channel=3Blue1Brown)
-[video2](https://www.youtube.com/watch?v=tIeHLnjs5U8&ab_channel=3Blue1Brown)

In [ ]:
%%add_to Network
def backprop(self, x, y):
    """Metoda vraca uredjeni par ``(nabla_b, nabla_w)`` koji predstavlja
    gradijente funkcije gubitka L za dani primjer (x,y).
    ``nabla_b`` i ``nabla_w`` su liste čiji su elementi arrayevi,
    slični ``self.biases`` i ``self.weights``."""

    nabla_b = [np.zeros(b.shape) for b in self.biases]
    nabla_w = [np.zeros(w.shape) for w in self.weights]

    # feedforward
    activation = x
    activations = [x]                               # lista u koju spremamo aktivacije po slojevima
    zs = []                                         # lista u koju spremamo dobivene z vrijednosti po slojevima
    for b, w in zip(self.biases, self.weights):
        z = np.dot(w, activation) + b
        zs.append(z)
        activation = sigmoid(z)
        activations.append(activation)

    # backward pass --> idemo sada od zadnjeg prema prvom sloju
    delta = (self.cost).delta(zs[-1], activations[-1], y) * sigmoid_derivacija(zs[-1])
    nabla_b[-1] = delta
    nabla_w[-1] = np.dot(delta, activations[-2].transpose())

    # Za l = 1 dobijemo zadnji sloj mreže, za l = 2 predzadnji, itd.
    for l in range(2, self.num_layers):
        z = zs[-l]
        sp = sigmoid_derivacija(z)
        delta = np.dot(self.weights[-l+1].transpose(), delta) * sp
        nabla_b[-l] = delta
        nabla_w[-l] = np.dot(delta, activations[-l-1].transpose())
    return (nabla_b, nabla_w)

#### One-hot encoder

In [ ]:
def one_hot(labela):
  x = np.zeros((10, 1))
  x[labela] = 1.0
  return x

# Trenirajmo mrežu

In [ ]:
import tensorflow as tf

In [ ]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

Transformacija podataka u prikladan oblik za treniranje mreže

In [ ]:
train_data = list(zip(X_train, y_train))
train_data = [(np.reshape(x, (28*28, 1)), one_hot(y)) for x, y in train_data]

In [ ]:
test_data = list(zip(X_test, y_test))
test_data = [(np.reshape(x, (28*28, 1)), one_hot(y)) for x, y in test_data]

In [ ]:
mreza = Network([784, 30, 10])

In [ ]:
# Vrijednosti hiperparametara
epochs = 15
mini_batch_size = 64
learning_rate = 1.0

In [ ]:
time_start = time()
mreza.fit(train_data, epochs, mini_batch_size, learning_rate, test_data)
time_end = time()
time_elapsed = time_end - time_start

print('\nTreniranje je trajalo {0} sekundi ({1} sekundi po epohi)!\n'.format(f'{time_elapsed:.0f}', f'{time_elapsed / epochs:.1f}'))
print('Točnost na skupu za testiranje {0}%'.format(100.0 * mreza.evaluate(test_data) / len(test_data)))

Treniranje mreže započelo...

##################################################
U tijeku EPOHA 1
accuracy treniranje 89.97%
accuracy testiranje 90.34%

##################################################
U tijeku EPOHA 2
accuracy treniranje 91.36%
accuracy testiranje 91.67%

##################################################
U tijeku EPOHA 3
accuracy treniranje 92.38%
accuracy testiranje 92.40%

##################################################
U tijeku EPOHA 4
accuracy treniranje 92.92%
accuracy testiranje 92.85%

##################################################
U tijeku EPOHA 5
accuracy treniranje 93.54%
accuracy testiranje 93.40%

##################################################
U tijeku EPOHA 6
accuracy treniranje 93.93%
accuracy testiranje 93.75%

##################################################
U tijeku EPOHA 7
accuracy treniranje 94.17%
accuracy testiranje 93.93%

##################################################
U tijeku EPOHA 8
accuracy treniranje 94.55%
accuracy testi

### Implementirajmo sada isti model u *Kerasu*

In [ ]:
jedinicna_normalna = tf.keras.initializers.RandomNormal(mean=0.0, stddev=1.0, seed=7)
he_normalna = tf.keras.initializers.HeNormal(seed=7)

In [ ]:
model = tf.keras.models.Sequential()

model.add(tf.keras.layers.Input(shape=(28, 28)))
model.add(tf.keras.layers.Flatten())

model.add(tf.keras.layers.Dense(30, activation='sigmoid',
                kernel_initializer=he_normalna,
                bias_initializer=jedinicna_normalna))

model.add(tf.keras.layers.Dense(10, activation='sigmoid',
                kernel_initializer=he_normalna,
                bias_initializer=jedinicna_normalna))

In [ ]:
model.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              metrics=['accuracy'])

In [ ]:
time_start = time()
povijest = model.fit(X_train,
                     tf.keras.utils.to_categorical(y_train),
                     epochs=epochs,
                     batch_size=mini_batch_size,
                     validation_data=(X_test, tf.keras.utils.to_categorical(y_test)))
time_end = time()

Epoch 1/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8963 - loss: 0.3576 - val_accuracy: 0.9376 - val_loss: 0.2134
Epoch 2/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9443 - loss: 0.1915 - val_accuracy: 0.9409 - val_loss: 0.1882
Epoch 3/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9542 - loss: 0.1538 - val_accuracy: 0.9535 - val_loss: 0.1497
Epoch 4/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9609 - loss: 0.1327 - val_accuracy: 0.9578 - val_loss: 0.1410
Epoch 5/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9653 - loss: 0.1183 - val_accuracy: 0.9577 - val_loss: 0.1315
Epoch 6/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9683 - loss: 0.1073 - val_accuracy: 0.9589 - val_loss: 0.1395
Epoch 7/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9707 - loss: 0.0986 - val_accuracy: 0.9622 - val_loss: 0.1246
Epoch 8/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9728 - loss: 0.0916 - val_accuracy: 0.

In [ ]:
time_elapsed = time_end - time_start
print('\nTreniranje je trajalo {0} sekundi ({1} sekundi po epohi)!\n'.format(f'{time_elapsed:.0f}', f'{time_elapsed / epochs:.1f}'))


Treniranje je trajalo 46 sekundi (3.1 sekundi po epohi)!



In [ ]:
_, test_acc = model.evaluate(X_test, tf.keras.utils.to_categorical(y_test), verbose=0)
print('Točnost na skupu za testiranje {0:.2f}%'.format(100.0 * test_acc))

Točnost na skupu za testiranje 96.59%


# 📝 **Zadaci za samostalan rad**
---


### **Zadatak 1. Usporedba načina inicijalizacija težina**

Promijenite način inicijalizacije težina tako da se za incijalizaciju težina koriste navedene inicijalizacije težina:

   **(a)** jedinična normalna distribucija $X\sim \mathcal{N}(0,1)$

  **(b)** normalna distribucija $X\sim \mathcal{N}(0,\frac{1}{n_{in}})$ (**LeCun** normalna inicijalizacija).

Nacrtajte graf koji će prikazivati kako se točnost mijenja za vrijeme treniranja; na istom grafu prikažite krivulje točnosti za He normalnu inicijalizaciju i inicijalizacije  **a** i **b**.




### **Zadatak 2. Izračun predviđanja mreže matričnim operacijama**

„Na ruke” izračunajte predviđanja neuronske mreže trenirane koristeći biblioteku **Keras** za sve primjere iz skupa za testiranje. Izračunajte točnost klasifikacije i usporedite je s rezultatom dobivenim metodom `evaluate`.


>**Što znači "na ruke"?** <br> Dohvatite naučene težine i pristranosti (*bias*) te <mark> **napišite vlastiti kod**</mark>  koji matričnim operacijama i primjenom aktivacijskih funkcija računa predviđanje mreže.


**NAPOMENA:**
*Točnost izračunata "na ruke" trebala bi se podudarati s točnošću dobivenom metodom `evaluate`, uz moguća mala odstupanja zbog numeričke preciznosti.*